# Retrieval playground

Rulează pipeline-ul de retrieval (query rewrite -> hybrid search semantic + BM25 -> fusion -> rerank -> top-K) pe întrebări ad-hoc, și inspectează rezultatele la fiecare etapă.

Presupune că `uv run python ingest.py` a fost deja rulat (există `db/`).

In [1]:
from dotenv import load_dotenv

load_dotenv(override=True)

# from config import FINAL_K, RETRIEVAL_K
from app.rag_simple.retrieval.pipeline import retrieve_with_trace, retrieve
from app.rag_simple.generation.answer import answer_question, make_messages

FINAL_K = 20
RETRIEVAL_K = 20

print(f"RETRIEVAL_K={RETRIEVAL_K}  FINAL_K={FINAL_K}")


/home/alex/projects/RAG/full-stack-fastapi-template/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:42: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


RETRIEVAL_K=20  FINAL_K=20


## Întrebări de test

Editează lista de mai jos cu întrebările pe care vrei să le testezi ad-hoc.

In [2]:
QUESTIONS = [
    # "cum se tratează cancerul cu macrobiotica?",
    "ce este echilibrul Yin-Yang?",
    # "ce rol au organele in psihologia macrobiotica?",
]


## Rulare + inspecție pe etape

In [3]:
def show_chunk(i, doc, show_context=False):
    source = doc.metadata.get("source", "?")
    heading = doc.metadata.get("h4") or doc.metadata.get("h3") or ""
    text = doc.page_content if show_context else doc.metadata.get("original_text", doc.page_content)
    preview = text.strip().replace("\n", " ")
    if len(preview) > 220:
        preview = preview[:220] + "..."
    print(f"[{i:>2}] {source} | {heading}\n     {preview}\n")


def show_trace(trace, top_n=10):
    print("=" * 100)
    print(f"QUESTION:   {trace.question}")
    print(f"REWRITTEN:  {trace.rewritten_question}")
    print(f"counts -> semantic={len(trace.semantic_hits)}  lexical={len(trace.lexical_hits)}  "
          f"fused={len(trace.fused)}  reranked={len(trace.reranked)}  final={len(trace.final)}")

    print("\n--- semantic hits (top 5) ---")
    for i, doc in enumerate(trace.semantic_hits[:5], start=1):
        show_chunk(i, doc)

    print("--- lexical / BM25 hits (top 5) ---")
    for i, doc in enumerate(trace.lexical_hits[:5], start=1):
        show_chunk(i, doc)

    print(f"--- final, reranked, top {top_n} (of {len(trace.final)}) ---")
    for i, doc in enumerate(trace.final[:top_n], start=1):
        show_chunk(i, doc)


In [4]:
traces = {}
for question in QUESTIONS:
    traces[question] = retrieve_with_trace(question)
    show_trace(traces[question])


QUESTION:   ce este echilibrul Yin-Yang?
REWRITTEN:  ce este echilibrul Yin-Yang?
counts -> semantic=20  lexical=20  fused=25  reranked=25  final=20

--- semantic hits (top 5) ---
[ 1] 04 seminar macrobiotica - tratamentul cancerului.md | Tratamentul cancerului
     nu le amestecăm. Dacă cancerul are o cauză Yang, trebuie să stopăm alimentele Yang. Nu trebuie să stopăm și alimentele Yin pentru că există același fenomen de atracție.

[ 2] 04 seminar macrobiotica - tratamentul cancerului.md | Tratamentul cancerului
     Când cancerul se răspândește, trebuie să știm să-l vindecăm. Știm că există cauze Yang și cauze Yin ale cancerului. Cauzele Yin sunt produsele chimice, zahărul, băuturile dulci, fructele, sucurile de fructe, ciocolata și...

[ 3] 04 seminar macrobiotica - tratamentul cancerului.md | Tratamentul cancerului
     Proporția regimului trebuie să fie: cereale complete 50%, un procent mic de supă de 5%, legume 20%, iar leguminoase și alge 15%. Cantitatea legumelor poate fi cresc

In [ ]:
traces = {}
history = []
for question in QUESTIONS:
    answer, chunks = answer_question(question, history)

print(answer)
print(chunks)

Echilibrul Yin-Yang este un principiu fundamental în filosofia și practica macrobioticii, care descrie interdependența și complementaritatea a două forțe opuse, dar interconectate, numite Yin și Yang. Aceste forțe reprezintă aspecte opuse ale realității, cum ar fi rece și cald, pasiv și activ, interior și exterior, dar care se influențează reciproc și trebuie să fie în echilibru pentru a menține sănătatea și armonia.

În contextul macrobioticii și al tratamentului cancerului, echilibrul Yin-Yang se referă la modul în care diferitele alimente, organe și stări energetice sunt clasificate ca Yin (mai receptive, mai pasive, mai umede) sau Yang (mai active, mai uscate, mai calde). De exemplu:

- Cancerul de tip Yin este asociat cu cauze alimentare Yin (produse chimice, zahăr, fructe, băuturi dulci) și se manifestă prin rupturi sau ulcerații.
- Cancerul de tip Yang este asociat cu cauze alimentare Yang (carne, ouă, sare în exces) și se manifestă prin tumori.

Pentru a trata sau preveni cance

## Inspecție liberă

Refolosește `traces[<întrebare>]` pentru a inspecta un obiect `RetrievalTrace` complet (`.semantic_hits`, `.lexical_hits`, `.fused`, `.reranked`, `.final`), sau rulează o întrebare nouă direct:

```python
trace = retrieve_with_trace("întrebarea mea ad-hoc")
show_trace(trace, top_n=20)
```